# DS2002 · ETL Concepts and Walmart Case Framing

**Lecture — 2026-10-12 · Fall 2026**  
**Class time:** 45 minutes

---

## Everything you have built is one pipeline

Seven weeks of separate skills are actually three stages, and you have been doing all three without naming them.

| Stage | What it means | What you already know |
|---|---|---|
| **Extract** | Get the data out of wherever it lives | `read_csv`, `read_sql_query`, `requests.get` with retries |
| **Transform** | Make it correct, consistent, and joined | dedupe, coerce types, normalize text, merge, validate |
| **Load** | Put the result somewhere the next person can use | `to_sql`, `to_csv`, a clean table with a documented schema |

The midterm and the capstone are both ETL pipelines. Today is about the properties that separate a pipeline you can run every week from a notebook that happened to work once.

### Four properties of a pipeline you can trust

**1. It is idempotent.** Run it twice, get the same result. You built this habit in week two: never mutate the source, always derive.

**2. It validates at the boundaries.** Row counts and assertions after every extract and every join. Failures should surface where they happen, not three cells later in a chart.

**3. It fails loudly.** No bare `except`. A pipeline that returns an empty frame on error is worse than one that crashes, because the empty frame becomes a finding.

**4. It records its decisions.** Which rows you dropped, which SKUs you merged, which assumptions you made. This is the difference between an analysis and an assertion.

Keep those four in view — they are the rubric for both projects, and for the rest of today.

### The Walmart case

In the run-up to Hurricane Frances in 2004, Walmart's CIO went looking through years of point-of-sale data from previous storms to answer a concrete question: what do people actually buy when a hurricane is coming?

The expected answers were there — flashlights, bottled water. The surprise was **strawberry Pop-Tarts, selling at roughly seven times their normal rate** before a storm, and beer running close behind. Nobody would have guessed it, and no amount of domain expertise would have produced it. It came out of the data.

Read `Walmart Case.pdf` in this folder before Monday. It frames your **midterm**, which starts next week.

Two things to hold onto while you read:

- The finding was only possible because the data was **consolidated**. Pop-Tarts appear in that system under many SKUs, in many spellings, across thousands of stores. Fragmented, the signal is invisible.
- The finding was only *useful* because someone acted on it — trucks rerouted, shelves restocked ahead of landfall. An analysis that changes no decision is a hobby.

### The consolidation problem, in miniature

Here is why fragmentation hides a 7x signal. Four transactions, one product, and one double-scanned row.

In [ ]:
import pandas as pd

rows = pd.DataFrame({
    'store': ['FL-105', 'FL-239', 'FL-105', 'FL-105'],
    'sku': ['PT-12', 'SKU#459812', 'POPTART-STRAW', 'PT-12'],
    'name': ['pop-tarts strawberry', 'Strawberry Pop-Tart',
             'Pop-Tarts Strawberry', 'pop-tarts strawberry'],
    'qty': [40, 15, 22, 40],
})
rows

In [ ]:
# The fragmented view: looks like three unremarkable products
print(rows.groupby('sku')['qty'].sum().sort_values(ascending=False))
print()
print('largest single SKU:', rows.groupby('sku')['qty'].sum().max())

Forty units. Against a normal week that is noise, and nobody reroutes a truck for it.

Now consolidate — and note the order of operations, because getting it backwards is a classic error. Deduplicate **first**; if you map to a canonical SKU before dropping the double-scanned row, the duplicate stops being detectable as a duplicate.

In [ ]:
CANON = {
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'POPTART-STRAW': 'POPTART-STRAW',
}

clean = rows.drop_duplicates()                    # dedupe FIRST
clean = clean.assign(canonical=clean['sku'].map(CANON))
print('rows:', len(rows), '->', len(clean))
print()
print(clean.groupby('canonical')['qty'].sum())

In [ ]:
fragmented = rows.groupby('sku')['qty'].sum().max()
consolidated = clean.groupby('canonical')['qty'].sum().max()
print(f'biggest number you can see fragmented:   {fragmented}')
print(f'biggest number after consolidation:      {consolidated}')
print(f'the signal was {consolidated / fragmented:.2f}x larger than it looked')

Seventy-seven units, not forty. Same data, and the difference is entirely in whether you noticed the product was the same product.

**Discussion — take two minutes with the person next to you:** at what point does this stop being a data-cleaning task and become a business judgment? `PT-12` and `POPTART-STRAW` are clearly the same. What about strawberry versus blueberry Pop-Tarts — one product or two? What about a 6-pack versus a 12-pack of the same flavor?

There is no correct answer, and that is the point. The mapping table is where you write your answer down so somebody can disagree with it.

### Where the mapping actually comes from

You will not hand-type a dictionary for a real catalog with thousands of products. You build a **candidate list** from the data, then a human confirms it.

This is the workflow you will use on the midterm: narrow with code, decide with judgment.

In [ ]:
catalog = pd.DataFrame({
    'sku': ['PT-12', 'SKU#459812', 'POPTART-STRAW', 'PT-BLUE-6',
            'WATER-24', 'BTL-WATER-24', 'FLASH-AA'],
    'name': ['pop-tarts strawberry', 'Strawberry Pop-Tart',
             'Pop-Tarts Strawberry', 'Pop-Tarts Blueberry 6ct',
             'Bottled Water 24pk', 'bottled water 24 pack', 'Flashlight AA'],
})

# Narrow with code: group by a normalized name and look for the collisions
catalog['normalized'] = (catalog['name'].str.lower()
                         .str.replace(r'[^a-z ]', '', regex=True)
                         .str.split().apply(lambda w: ' '.join(sorted(w))))

candidates = (catalog.groupby('normalized')['sku']
              .agg(list).loc[lambda s: s.str.len() > 1])
candidates

Sorting the words catches `bottled water 24 pack` and `Bottled Water 24pk` as the same thing. It does not catch `PT-12` and `POPTART-STRAW`, whose names differ in word order only — and it correctly leaves blueberry alone.

No automated rule gets this fully right. The point is that code reduces thousands of SKUs to a few dozen judgment calls, and then a person makes them.

### Load: the stage everyone skips

Extract and transform get all the attention. Load is what makes the work reusable — the clean table becomes the input to every question that comes after, and nobody re-runs your cleaning to answer it.

For this course, loading means writing a clean table into SQLite and querying it back to prove it landed. Wednesday you will do this yourself.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
out = clean[['store', 'canonical', 'qty']]
out.to_sql('clean_sales', conn, index=False, if_exists='replace')

# Read it back -- always verify the load rather than assuming it
check = pd.read_sql_query(
    'SELECT canonical, SUM(qty) AS units FROM clean_sales GROUP BY canonical', conn)
print(check)
print()
print('rows written:', pd.read_sql_query(
    'SELECT COUNT(*) AS n FROM clean_sales', conn)['n'][0], 'of', len(out))

### What next week looks like

Monday you get the real Walmart data: hundreds of thousands of transactions, fragmented SKUs, three timestamp formats, negative quantities, store ids in four different casings. You will work in teams of two to four for three weeks, and the deliverable is a pipeline plus five answered questions plus a recommendation.

Wednesday's studio is the dress rehearsal: a full extract, transform, and load on a small sample of that exact data. Friday's lab gets you into the real catalog.

Read the case before Monday. The teams that do well are the ones that understand the business question before they start writing pandas.